In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data/processed/electricity_clean.csv")

data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)

data = data.sort_values("timestamp")
data.reset_index(drop=True, inplace=True)

data.head()

,timestamp,load,period
0,2017-12-31 23:00:00+00:00,14978.54,calm
1,2018-01-01 00:00:00+00:00,14397.65,calm
2,2018-01-01 01:00:00+00:00,13789.46,calm
3,2018-01-01 02:00:00+00:00,13434.45,calm
4,2018-01-01 03:00:00+00:00,13285.24,calm


In [3]:
data["hour"] = data["timestamp"].dt.hour
data["day_of_week"] = data["timestamp"].dt.dayofweek  # 0=Monday
data["month"] = data["timestamp"].dt.month

data.head()

,timestamp,load,period,hour,day_of_week,month
0,2017-12-31 23:00:00+00:00,14978.54,calm,23,6,12
1,2018-01-01 00:00:00+00:00,14397.65,calm,0,0,1
2,2018-01-01 01:00:00+00:00,13789.46,calm,1,0,1
3,2018-01-01 02:00:00+00:00,13434.45,calm,2,0,1
4,2018-01-01 03:00:00+00:00,13285.24,calm,3,0,1


In [4]:
data["is_weekend"] = data["day_of_week"].isin([5, 6]).astype(int)

data.head()

,timestamp,load,period,hour,day_of_week,month,is_weekend
0,2017-12-31 23:00:00+00:00,14978.54,calm,23,6,12,1
1,2018-01-01 00:00:00+00:00,14397.65,calm,0,0,1,0
2,2018-01-01 01:00:00+00:00,13789.46,calm,1,0,1,0
3,2018-01-01 02:00:00+00:00,13434.45,calm,2,0,1,0
4,2018-01-01 03:00:00+00:00,13285.24,calm,3,0,1,0


In [5]:
def get_season(month):
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"

data["season"] = data["month"].apply(get_season)

data.head()

,timestamp,load,period,hour,day_of_week,month,is_weekend,season
0,2017-12-31 23:00:00+00:00,14978.54,calm,23,6,12,1,winter
1,2018-01-01 00:00:00+00:00,14397.65,calm,0,0,1,0,winter
2,2018-01-01 01:00:00+00:00,13789.46,calm,1,0,1,0,winter
3,2018-01-01 02:00:00+00:00,13434.45,calm,2,0,1,0,winter
4,2018-01-01 03:00:00+00:00,13285.24,calm,3,0,1,0,winter


In [6]:
season_map = {
    "winter": 0,
    "spring": 1,
    "summer": 2,
    "autumn": 3
}

data["season_encoded"] = data["season"].map(season_map)

data.head()

,timestamp,load,period,hour,day_of_week,month,is_weekend,season,season_encoded
0,2017-12-31 23:00:00+00:00,14978.54,calm,23,6,12,1,winter,0
1,2018-01-01 00:00:00+00:00,14397.65,calm,0,0,1,0,winter,0
2,2018-01-01 01:00:00+00:00,13789.46,calm,1,0,1,0,winter,0
3,2018-01-01 02:00:00+00:00,13434.45,calm,2,0,1,0,winter,0
4,2018-01-01 03:00:00+00:00,13285.24,calm,3,0,1,0,winter,0


In [7]:
data["lag_1"] = data["load"].shift(1)
data["lag_24"] = data["load"].shift(24)
data["lag_168"] = data["load"].shift(168)

data.head()

,timestamp,load,period,hour,day_of_week,month,is_weekend,season,season_encoded,lag_1,lag_24,lag_168
0,2017-12-31 23:00:00+00:00,14978.54,calm,23,6,12,1,winter,0,NaN,NaN,NaN
1,2018-01-01 00:00:00+00:00,14397.65,calm,0,0,1,0,winter,0,14978.54,NaN,NaN
2,2018-01-01 01:00:00+00:00,13789.46,calm,1,0,1,0,winter,0,14397.65,NaN,NaN
3,2018-01-01 02:00:00+00:00,13434.45,calm,2,0,1,0,winter,0,13789.46,NaN,NaN
4,2018-01-01 03:00:00+00:00,13285.24,calm,3,0,1,0,winter,0,13434.45,NaN,NaN


In [8]:
data["rolling_mean_24"] = data["load"].rolling(window=24).mean()

data.head()

,timestamp,load,period,hour,day_of_week,month,is_weekend,season,season_encoded,lag_1,lag_24,lag_168,rolling_mean_24
0,2017-12-31 23:00:00+00:00,14978.54,calm,23,6,12,1,winter,0,NaN,NaN,NaN,NaN
1,2018-01-01 00:00:00+00:00,14397.65,calm,0,0,1,0,winter,0,14978.54,NaN,NaN,NaN
2,2018-01-01 01:00:00+00:00,13789.46,calm,1,0,1,0,winter,0,14397.65,NaN,NaN,NaN
3,2018-01-01 02:00:00+00:00,13434.45,calm,2,0,1,0,winter,0,13789.46,NaN,NaN,NaN
4,2018-01-01 03:00:00+00:00,13285.24,calm,3,0,1,0,winter,0,13434.45,NaN,NaN,NaN


In [9]:
data = data.dropna()

data.reset_index(drop=True, inplace=True)

data.head()

,timestamp,load,period,hour,day_of_week,month,is_weekend,season,season_encoded,lag_1,lag_24,lag_168,rolling_mean_24
0,2018-01-07 23:00:00+00:00,15993.19,calm,23,6,1,1,winter,0,16836.21,15330.00,14978.54,17287.211250
1,2018-01-08 00:00:00+00:00,15633.59,calm,0,0,1,0,winter,0,15993.19,14662.44,14397.65,17327.675833
2,2018-01-08 01:00:00+00:00,15462.54,calm,1,0,1,0,winter,0,15633.59,14367.65,13789.46,17373.296250
3,2018-01-08 02:00:00+00:00,15445.81,calm,2,0,1,0,winter,0,15462.54,14295.76,13434.45,17421.215000
4,2018-01-08 03:00:00+00:00,15892.14,calm,3,0,1,0,winter,0,15445.81,14252.75,13285.24,17489.522917


In [10]:
model_data = data[[
    "timestamp",
    "load",
    "period",  # for evaluation only
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "season",
    "season_encoded",
    "lag_1",
    "lag_24",
    "lag_168",
    "rolling_mean_24"
]]

model_data.head()

,timestamp,load,period,hour,day_of_week,month,is_weekend,season,season_encoded,lag_1,lag_24,lag_168,rolling_mean_24
0,2018-01-07 23:00:00+00:00,15993.19,calm,23,6,1,1,winter,0,16836.21,15330.00,14978.54,17287.211250
1,2018-01-08 00:00:00+00:00,15633.59,calm,0,0,1,0,winter,0,15993.19,14662.44,14397.65,17327.675833
2,2018-01-08 01:00:00+00:00,15462.54,calm,1,0,1,0,winter,0,15633.59,14367.65,13789.46,17373.296250
3,2018-01-08 02:00:00+00:00,15445.81,calm,2,0,1,0,winter,0,15462.54,14295.76,13434.45,17421.215000
4,2018-01-08 03:00:00+00:00,15892.14,calm,3,0,1,0,winter,0,15445.81,14252.75,13285.24,17489.522917


In [11]:
model_data.to_csv("../data/processed/electricity_features.csv", index=False)

print("Feature engineering complete. File saved.")

Feature engineering complete. File saved.
